In [14]:
import os
from dotenv import load_dotenv
from datetime import date, timedelta
import requests
import zipfile
import io
import logging
from datetime import date, timedelta
import pandas as pd
import numpy as np
import yfinance as yfinance
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from edinet_xbrl.edinet_xbrl_parser import EdinetXbrlParser

load_dotenv()
EDINET_API_KEY = os.getenv('EDINET_API_KEY')

In [23]:
def get_documents_by_date(target_date, doc_type='030000'):
    """
    指定した日付にEDINETで開示された書類一覧を取得し、
    指定doc_type(有報)を満たすdoc_idとedinet_codeのリストを返す。
    """
    if isinstance(target_date, date):
        date_str = target_date.strftime("%Y-%m-%d")
    else:
        date_str = target_date
    base_url = "https://disclosure.edinet-fsa.go.jp/api/v2/documents.json"
    params = {'date': date_str, 'type': 2,'Subscription-Key':EDINET_API_KEY}
    r = requests.get(base_url, params=params)
    r.raise_for_status()
    data = r.json()
    results = data.get('results', [])

    docs = []


    for d in results:
        if d.get('formCode') == doc_type and d.get('docTypeCode') == '120':
            docs.append(d)

    return docs

h = get_documents_by_date(date(2024, 4, 12),doc_type='030000')

In [24]:
doc_id = h[0]['docID']
periodEnd_date= date.fromisoformat(h[0]['periodEnd'])

In [25]:
h

[{'seqNumber': 284,
  'docID': 'S100T6W5',
  'edinetCode': 'E06017',
  'secCode': None,
  'JCN': None,
  'filerName': '欧州評議会開発銀行',
  'fundCode': None,
  'ordinanceCode': '020',
  'formCode': '030000',
  'docTypeCode': '120',
  'periodStart': '2023-01-01',
  'periodEnd': '2023-12-31',
  'submitDateTime': '2024-04-12 14:50',
  'docDescription': '有価証券報告書',
  'issuerEdinetCode': None,
  'subjectEdinetCode': None,
  'subsidiaryEdinetCode': None,
  'currentReportReason': None,
  'parentDocID': None,
  'opeDateTime': None,
  'withdrawalStatus': '0',
  'docInfoEditStatus': '0',
  'disclosureStatus': '0',
  'xbrlFlag': '0',
  'pdfFlag': '1',
  'attachDocFlag': '1',
  'englishDocFlag': '0',
  'csvFlag': '0',
  'legalStatus': '1'}]

In [18]:
import os
import io
import zipfile
import tempfile
import requests
import logging

def download_xbrl_file(doc_id,output_dir = "./xbrl/"):
    """

    """
    path = output_dir + doc_id + '/'
    
    base_url = "https://disclosure.edinet-fsa.go.jp/api/v2/documents"
    params = {
        'type': 1,
        'Subscription-Key': EDINET_API_KEY
    }
    
    try:
        response = requests.get(f"{base_url}/{doc_id}", params=params, stream=True)
        response.raise_for_status()
    except Exception as e:
        logging.error("EDINET APIからのダウンロードに失敗しました: %s", e)
        return None
    
    # try:
    # ダウンロードしたZIPアーカイブをメモリ上で読み込む
    
    z = zipfile.ZipFile(io.BytesIO(response.content))
    # 出力先ディレクトリの作成
    if not os.path.exists(path):
        os.makedirs(path)
            
    filename = doc_id + ".zip"
    with open(path+filename, 'wb') as f:    
        for chunk in response.iter_content(chunk_size=1024):
          f.write(chunk)

    with zipfile.ZipFile(path+filename) as zip_f:
        zip_f.extractall(path)


        # ZIP 内で拡張子が .xbrl のファイルを検索（大文字小文字区別しない）
        xbrl_files = [f for f in zip_f.namelist() if f.lower().endswith('.xbrl')]
        if not xbrl_files:
            logging.warning("doc_id=%s のZIP内にXBRLファイルが見つかりませんでした", doc_id)
            return None
        
        # 例として最初に見つかった XBRL ファイルの絶対パスを返す
        xbrl_file_rel_path = xbrl_files[0]
        current_directory = os.getcwd()
        xbrl_file_abs_path = os.path.join(current_directory + path[1:], xbrl_file_rel_path)
        
    return xbrl_file_abs_path
    

In [19]:
path = download_xbrl_file(doc_id='S100T6W5',output_dir = "./xbrl/")


In [15]:
xbrl_file_abs_path = download_xbrl_file(doc_id)

In [20]:
import os
import pandas as pd
from arelle import Cntlr
from datetime import datetime

def extract_financial_data(xbrl_file):
    # Arelle のコントローラ作成（ログ出力は標準出力に設定）
    cntlr = Cntlr.Cntlr(logFileName='logToPrint')
    
    # XBRL ファイルを読み込み
    modelXbrl = cntlr.modelManager.load(xbrl_file)
    
    # 抽出データを格納するリスト
    data = []
    
    # 各 fact から情報を抽出
    for fact in modelXbrl.facts:
        # 必要な情報例: コンセプト、値、単位、コンテキストID
        row = {
            'concept': fact.concept.qname.localName,
            'concept_jp': fact.concept.label(preferredLabel=None, lang='ja', linkroleHint=None), 
            'value': fact.value,
            'unit': fact.unitID if fact.unitID else '',
            'context': fact.contextID,
        }
        data.append(row)
    
    # リストを pandas DataFrame に変換
    df = pd.DataFrame(data)
    return df

def get_year_from_context(context_str, periodEnd_date):
    """
    context の文字列に含まれるキーワードに基づいて、
    対象期を periodEnd から何年前か計算し、'YYYY' 形式で返す。
    """
    if "CurrentYear" in context_str:
        target_date = periodEnd_date
    elif "Prior1Year" in context_str:
        target_date = periodEnd_date.replace(year=periodEnd_date.year - 1)
    elif "Prior2Year" in context_str:
        target_date = periodEnd_date.replace(year=periodEnd_date.year - 2)
    elif "Prior3Year" in context_str:
        target_date = periodEnd_date.replace(year=periodEnd_date.year - 3)
    elif "Prior4Year" in context_str:
        target_date = periodEnd_date.replace(year=periodEnd_date.year - 4)
    else:
        # 対象外の場合は None を返す
        return None
    
    return target_date.strftime("%Y")

def is_non_consolidated(context_str):
    """
    context の文字列に含まれるキーワードに基づいて、
    連結か非連結（単独）かを判断する。
    """
    return "NonConsolidatedMember" in context_str

def process_xbrl_file(xbrl_file_path, periodEnd_date_str):
    # ファイルの存在確認
    if not os.path.exists(xbrl_file_path):
        print("XBRL ファイルが見つかりません:", xbrl_file_path)
        return None
    
    # XBRLデータの抽出
    df = extract_financial_data(xbrl_file_path)
    
    # 日付文字列をdateオブジェクトに変換
    periodEnd_date = datetime.strptime(periodEnd_date_str, "%Y-%m-%d").date()
    
    # 各行のコンテキストに対して年度と連結/非連結情報を追加
    df['year'] = df['context'].apply(lambda x: get_year_from_context(x, periodEnd_date))
    df['is_non_consolidated'] = df['context'].apply(is_non_consolidated)
    
    return df


xbrl_path = xbrl_file_abs_path # 解析対象の XBRL ファイルパスに変更
periodEnd_date = "2024-03-31"

result_df = process_xbrl_file(xbrl_path, periodEnd_date)

if result_df is not None:
    # 結果の表示例
    print(f"データ行数: {len(result_df)}")
    print(f"年度別データ数: {result_df['year'].value_counts()}")
    print(f"連結/非連結データ数: {result_df['is_non_consolidated'].value_counts()}")


データ行数: 2253
年度別データ数: year
2024    1154
2023     646
2022      79
2020      49
2021      49
Name: count, dtype: int64
連結/非連結データ数: is_non_consolidated
False    1632
True      621
Name: count, dtype: int64


In [28]:
result_df[result_df['is_non_consolidated']==True]

,concept,concept_jp,value,unit,context,year,is_non_consolidated
101,OrdinaryIncomeSummaryOfBusinessResults,経常収益,73250000000,JPY,Prior4YearDuration_NonConsolidatedMember,2020,True
102,OrdinaryIncomeSummaryOfBusinessResults,経常収益,72610000000,JPY,Prior3YearDuration_NonConsolidatedMember,2021,True
103,OrdinaryIncomeSummaryOfBusinessResults,経常収益,86664000000,JPY,Prior2YearDuration_NonConsolidatedMember,2022,True
104,OrdinaryIncomeSummaryOfBusinessResults,経常収益,103401000000,JPY,Prior1YearDuration_NonConsolidatedMember,2023,True
105,OrdinaryIncomeSummaryOfBusinessResults,経常収益,110306000000,JPY,CurrentYearDuration_NonConsolidatedMember,2024,True
...,...,...,...,...,...,...,...
2236,NotesRegardingAmountsOfReductionEntryOfPropert...,有形固定資産の圧縮記帳額の注記,"<p class=""smt_text6"" style=""orphans:0;widows:0...",,CurrentYearInstant_NonConsolidatedMember,2024,True
2237,NotesRegardingGuaranteeObligationForCorporateB...,資産の部の社債に係る保証債務に関する注記,"<p class=""smt_text6"" style=""orphans:0;widows:0...",,CurrentYearInstant_NonConsolidatedMember,2024,True
2238,NotesRegardingGrossAmountOfAccountsReceivableT...,取締役及び監査役に対する金銭債務総額に関する注記,"<p class=""smt_text6"" style=""orphans:0;widows:0...",,CurrentYearDuration_NonConsolidatedMember,2024,True
2239,NotesRegardingPrincipalAmountOfTrustWithContra...,元本補填契約のある信託の元本金額に関する注記,"<p style=""page-break-before:always; line-heigh...",,CurrentYearDuration_NonConsolidatedMember,2024,True


### assets = df[df['unit']=='JPY']
# assets = assets.copy()
# assets['context_YYYY'] = assets['context'].apply(set_context_YYYY)
# assets['context_NCM'] = assets['context'].apply(set_context_NCM)

df = df.copy()
df['context_YYYY'] = df['context'].apply(set_context_YYYY)
df['context_NCM'] = df['context'].apply(set_context_NCM)

In [9]:
columns_to_check = ['concept_jp']

filtered_df = df[
    df[columns_to_check]
    .apply(lambda row: row.astype(str).str.contains("配当").any(), axis=1)
]

In [10]:
filtered_df

,concept,concept_jp,value,unit,context
161,DividendPaidPerShareSummaryOfBusinessResults,１株当たり配当額,40.00,JPYPerShares,Prior4YearDuration_NonConsolidatedMember
162,DividendPaidPerShareSummaryOfBusinessResults,１株当たり配当額,40.00,JPYPerShares,Prior3YearDuration_NonConsolidatedMember
163,DividendPaidPerShareSummaryOfBusinessResults,１株当たり配当額,80.00,JPYPerShares,Prior2YearDuration_NonConsolidatedMember
164,DividendPaidPerShareSummaryOfBusinessResults,１株当たり配当額,80.00,JPYPerShares,Prior1YearDuration_NonConsolidatedMember
165,DividendPaidPerShareSummaryOfBusinessResults,１株当たり配当額,90.00,JPYPerShares,CurrentYearDuration_NonConsolidatedMember
166,InterimDividendPaidPerShareSummaryOfBusinessRe...,１株当たり中間配当額,17.50,JPYPerShares,Prior4YearDuration_NonConsolidatedMember
167,InterimDividendPaidPerShareSummaryOfBusinessRe...,１株当たり中間配当額,17.50,JPYPerShares,Prior3YearDuration_NonConsolidatedMember
168,InterimDividendPaidPerShareSummaryOfBusinessRe...,１株当たり中間配当額,17.50,JPYPerShares,Prior2YearDuration_NonConsolidatedMember
169,InterimDividendPaidPerShareSummaryOfBusinessRe...,１株当たり中間配当額,40.00,JPYPerShares,Prior1YearDuration_NonConsolidatedMember
170,InterimDividendPaidPerShareSummaryOfBusinessRe...,１株当たり中間配当額,50.00,JPYPerShares,CurrentYearDuration_NonConsolidatedMember


In [11]:
from dotenv import load_dotenv
import os
from sshtunnel import SSHTunnelForwarder
from pymongo import MongoClient


# .envファイルをロードして環境変数を読み込む
load_dotenv()

# SSH接続情報（.envから読み込み）
SSH_HOST = os.getenv('SSH_HOST')
SSH_PORT = int(os.getenv('SSH_PORT', 22))
SSH_USERNAME = os.getenv('SSH_USERNAME')
SSH_PASSWORD = os.getenv('SSH_PASSWORD')

# MongoDB接続情報（.envから読み込み）
MONGO_HOST = os.getenv('MONGO_HOST', '127.0.0.1')
MONGO_PORT = int(os.getenv('MONGO_PORT', 27017))
DATABASE_NAME = 'test'
COLLECTION_NAME = 'collection'
    
def ssh_connection_and_write(df):
    tunnel = SSHTunnelForwarder(
        (SSH_HOST, SSH_PORT),
        ssh_username=SSH_USERNAME,
        ssh_password=SSH_PASSWORD,
        remote_bind_address=(MONGO_HOST, MONGO_PORT)
    )
    tunnel.start()
    try:
        local_port = tunnel.local_bind_port
        print(f"SSHトンネル確立: ローカルポート {local_port} がリモートの {MONGO_HOST}:{MONGO_PORT} にマッピングされました。")
        
        client = MongoClient('127.0.0.1', local_port)
        db = client[DATABASE_NAME]
        collection = db[COLLECTION_NAME]
        
        records = df.to_dict(orient='records')
        result = collection.insert_many(records)

    finally:
        tunnel.stop()
        
    return result.inserted_ids

ssh_connection_and_write(df)

SSHトンネル確立: ローカルポート 61717 がリモートの 127.0.0.1:27017 にマッピングされました。


[ObjectId('67cc53a0d432fef7e7eb21af'),
 ObjectId('67cc53a0d432fef7e7eb21b0'),
 ObjectId('67cc53a0d432fef7e7eb21b1'),
 ObjectId('67cc53a0d432fef7e7eb21b2'),
 ObjectId('67cc53a0d432fef7e7eb21b3'),
 ObjectId('67cc53a0d432fef7e7eb21b4'),
 ObjectId('67cc53a0d432fef7e7eb21b5'),
 ObjectId('67cc53a0d432fef7e7eb21b6'),
 ObjectId('67cc53a0d432fef7e7eb21b7'),
 ObjectId('67cc53a0d432fef7e7eb21b8'),
 ObjectId('67cc53a0d432fef7e7eb21b9'),
 ObjectId('67cc53a0d432fef7e7eb21ba'),
 ObjectId('67cc53a0d432fef7e7eb21bb'),
 ObjectId('67cc53a0d432fef7e7eb21bc'),
 ObjectId('67cc53a0d432fef7e7eb21bd'),
 ObjectId('67cc53a0d432fef7e7eb21be'),
 ObjectId('67cc53a0d432fef7e7eb21bf'),
 ObjectId('67cc53a0d432fef7e7eb21c0'),
 ObjectId('67cc53a0d432fef7e7eb21c1'),
 ObjectId('67cc53a0d432fef7e7eb21c2'),
 ObjectId('67cc53a0d432fef7e7eb21c3'),
 ObjectId('67cc53a0d432fef7e7eb21c4'),
 ObjectId('67cc53a0d432fef7e7eb21c5'),
 ObjectId('67cc53a0d432fef7e7eb21c6'),
 ObjectId('67cc53a0d432fef7e7eb21c7'),
 ObjectId('67cc53a0d432fe

In [12]:
records = df.to_dict(orient='records')
records

[{'concept': 'NumberOfSubmissionDEI',
  'concept_jp': '提出回数',
  'value': '1',
  'unit': 'pure',
  'context': 'FilingDateInstant'},
 {'concept': 'OrdinaryIncomeSummaryOfBusinessResults',
  'concept_jp': '経常収益',
  'value': '88871000000',
  'unit': 'JPY',
  'context': 'Prior4YearDuration'},
 {'concept': 'OrdinaryIncomeSummaryOfBusinessResults',
  'concept_jp': '経常収益',
  'value': '85715000000',
  'unit': 'JPY',
  'context': 'Prior3YearDuration'},
 {'concept': 'OrdinaryIncomeSummaryOfBusinessResults',
  'concept_jp': '経常収益',
  'value': '98306000000',
  'unit': 'JPY',
  'context': 'Prior2YearDuration'},
 {'concept': 'OrdinaryIncomeSummaryOfBusinessResults',
  'concept_jp': '経常収益',
  'value': '115289000000',
  'unit': 'JPY',
  'context': 'Prior1YearDuration'},
 {'concept': 'OrdinaryIncomeSummaryOfBusinessResults',
  'concept_jp': '経常収益',
  'value': '122630000000',
  'unit': 'JPY',
  'context': 'CurrentYearDuration'},
 {'concept': 'TrustFeesSummaryOfBusinessResults',
  'concept_jp': '信託報酬',
  

In [26]:
import asyncio
import os
import zipfile
import pandas as pd
from typing import Dict, List, Optional, Union
from playwright.async_api import async_playwright, Page
import io
import glob

async def download_edinet_file(
    url: str = "https://disclosure2.edinet-fsa.go.jp/weee0010.aspx",
    download_dir: str = None,
    js_function: str = "onDownloadEdinet()",
    timeout: int = 60000,
    extract: bool = True,
    headless: bool = True,
    encoding: str = "cp932"
) -> Dict[str, pd.DataFrame]:
    """EDINETからファイルをダウンロードし、CSVをデータフレームに変換します"""
    
    # ダウンロードディレクトリの設定
    if download_dir is None:
        download_dir = os.path.join(os.getcwd(), "downloads")
    os.makedirs(download_dir, exist_ok=True)
    
    # 結果格納用の辞書
    result_dataframes = {}
    
    try:
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=headless)
            context = await browser.new_context(accept_downloads=True)
            page = await context.new_page()
            
            # サイトにアクセスしてダウンロード
            await page.goto(url)
            download_path = await download_and_wait(page, download_dir, js_function, timeout)
            await browser.close()
            
            if download_path and os.path.exists(download_path):
                # ZIPファイルの処理
                with zipfile.ZipFile(download_path, 'r') as zip_ref:
                    if extract:
                        # ファイルを解凍してからCSVを読み込む
                        extract_dir = os.path.join(download_dir, "extracted")
                        os.makedirs(extract_dir, exist_ok=True)
                        zip_ref.extractall(path=extract_dir)
                        
                        # CSV検索とデータフレーム変換
                        for csv_file in find_csv_files(extract_dir):
                            df = read_csv_to_dataframe(csv_file, encoding)
                            if df is not None:
                                result_dataframes[os.path.basename(csv_file)] = df
                    else:
                        # ZIP内のCSVを直接読み込む
                        for csv_filename in [f for f in zip_ref.namelist() if f.lower().endswith('.csv')]:
                            with zip_ref.open(csv_filename) as csv_file:
                                content = csv_file.read()
                                try:
                                    df = pd.read_csv(io.BytesIO(content), encoding=encoding, skiprows=1, dtype=str)
                                    result_dataframes[os.path.basename(csv_filename)] = df
                                except Exception:
                                    # エンコーディングを変えて再試行
                                    try:
                                        df = pd.read_csv(io.BytesIO(content), encoding="utf-8", skiprows=1, dtype=str)
                                        result_dataframes[os.path.basename(csv_filename)] = df
                                    except Exception:
                                        pass
    except Exception as e:
        print(f"エラーが発生しました: {str(e)}")
    
    return result_dataframes

async def download_and_wait(page: Page, download_dir: str, js_function: str, timeout: int) -> Optional[str]:
    """ダウンロードを開始して完了を待つ"""
    try:
        # ダウンロードタスクを作成
        download_future = asyncio.create_task(page.wait_for_event('download', timeout=timeout))
        
        # JavaScriptを実行してダウンロード開始
        await page.evaluate(js_function)
        
        # ダウンロード完了を待つ
        download = await download_future
        save_path = os.path.join(download_dir, download.suggested_filename)
        await download.save_as(save_path)
        return save_path
    except Exception as e:
        print(f"ダウンロード中にエラーが発生しました: {str(e)}")
        return None

def find_csv_files(directory: str) -> List[str]:
    """指定ディレクトリ内のCSVファイルを検索"""
    return glob.glob(os.path.join(directory, "**", "*.csv"), recursive=True)

def read_csv_to_dataframe(csv_path: str, encoding: str = "cp932") -> Optional[pd.DataFrame]:
    """CSVファイルをデータフレームに読み込む（1行目をスキップ）"""
    try:
        try:
            df = pd.read_csv(csv_path, encoding=encoding, skiprows=1, dtype=str)
        except UnicodeDecodeError:
            df = pd.read_csv(csv_path, encoding="utf-8", skiprows=1, dtype=str)
        return df
    except Exception:
        return None

async def get_edinet_dataframes():
    """EDINETからCSVをダウンロードしてデータフレームを返す"""
    return await download_edinet_file(
        headless=True,  # 必要に応じてTrue/Falseを切り替え
        extract=True
    )

async def get_first_dataframe():
    """最初のCSVデータフレームのみを返す"""
    dataframes = await get_edinet_dataframes()
    if dataframes:
        # 最初のデータフレームを返す
        return next(iter(dataframes.values()))
    return pd.DataFrame()  # 空のデータフレームを返す

# Jupyter環境で実行する場合:
# すべてのデータフレームを辞書として取得
dfs = await get_edinet_dataframes()
dfs_tmp = dfs['EdinetcodeDlInfo.csv']
df = dfs_tmp[dfs_tmp['上場区分']=='上場']
df.loc[:, '証券コード'] = df['証券コード'].str.rstrip('0')

In [31]:
list(df.loc[:, '証券コード'])

['1376',
 '1377',
 '1375',
 '1379',
 '1381',
 '1382',
 '1911',
 '1301',
 '1332',
 '1333',
 '5711',
 '5713',
 '5706',
 '5729',
 '1491',
 '5714',
 '3315',
 '5715',
 '9675',
 '8835',
 '7021',
 '1515',
 '1518',
 '1662',
 '1605',
 '1925',
 '1801',
 '1803',
 '1802',
 '1861',
 '1812',
 '1887',
 '182',
 '181',
 '1815',
 '1882',
 '1884',
 '1811',
 '1941',
 '1942',
 '1944',
 '1946',
 '1892',
 '1885',
 '1888',
 '189',
 '1833',
 '1821',
 '1893',
 '1808',
 '1945',
 '1814',
 '1951',
 '1822',
 '1813',
 '1926',
 '1841',
 '1827',
 '1961',
 '1972',
 '1835',
 '1968',
 '1949',
 '195',
 '1847',
 '185',
 '1964',
 '1852',
 '1826',
 '1853',
 '1897',
 '1934',
 '1967',
 '1929',
 '1928',
 '1959',
 '186',
 '187',
 '1969',
 '1975',
 '8157',
 '1976',
 '8836',
 '1807',
 '1982',
 '196',
 '1866',
 '1898',
 '1867',
 '1939',
 '1992',
 '1979',
 '1938',
 '9767',
 '1981',
 '198',
 '1899',
 '1873',
 '1869',
 '1914',
 '193',
 '9743',
 '1966',
 '1973',
 '1878',
 '1879',
 '1994',
 '1965',
 '1952',
 '1905',
 '1904',
 '1848',
 '